# Lernen aus Feedback – Konzepte

**Geschäftliches Problem:** Eine Stellenanzeige auswählen, die besonders hervorgehoben werden soll, und gleichzeitig aus unsicheren Rückmeldungen lernen.

Ein Modell unterstützt eine Entscheidung; es trifft die geschäftliche Entscheidung nicht für uns.

## Lernziele

- Exploration (Ausprobieren) und Exploitation (Nutzen des bisherigen Wissens) erklären.

- Bandit-Verfahren von sequenziellen Entscheidungen unterscheiden, bei denen Handlungen den weiteren Verlauf beeinflussen.

- Überwachtes Lernen bei grossen Sprachmodellen (LLMs) von belohnungsbasiertem Training unterscheiden.

## Datensatz und praktischer Umfang

Alle Klicks und Klickraten sind simuliert. Das Beispiel mit dem Assistenten ist hypothetisch und stellt keinen tatsächlichen RL-Trainingsdurchlauf dar.

Der gesamte Code wird bereitgestellt. Führe die Zellen von oben nach unten aus.

## Bibliotheken und Einstellungen

Verwende den vorbereiteten Python-3.11-Kernel in Codespaces. Es werden weder eine GPU noch ein API-Schlüssel benötigt.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 7
np.random.seed(SEED)
DATA_ROOT = Path('../../../00_Data')

## 1. Ausgangsbasis: Nur die angezeigte Stellenanzeige liefert Feedback

Ein Arm ist eine verfügbare Auswahlmöglichkeit. Eine Belohnung (Reward) ist die festgelegte Reaktion, hier ein Klick. Eine Policy ist die Entscheidungsregel, nach der eine Auswahl getroffen wird.

Die feste Auswahl von Anzeige A und die zufällige Auswahl dienen als Vergleichsmethoden (Baselines). Sie stellen keine vollständigen A/B-Experimente dar.

In [2]:
display(pd.DataFrame({'ad': ['A', 'B'], 'observed clicks': [2, 1], 'displays': [10, 10]})
        .assign(observed_rate=lambda frame: frame['observed clicks'] / frame['displays']))

,ad,observed clicks,displays,observed_rate
0,A,2,10,0.2
1,B,1,10,0.1


## 2. Meistens das Bewährte nutzen, manchmal etwas Neues ausprobieren

Die Epsilon-Greedy-Strategie wählt bei ungefähr 10 % der Entscheidungen zufällig eine Option aus (Exploration). Ansonsten wählt sie die Option mit der höchsten geschätzten Erfolgsrate (Exploitation).

Auch bei der Exploration kann die aktuell führende Anzeige ausgewählt werden. Das bedeutet, dass nicht automatisch 10 % der Einblendungen auf die schwächere Anzeige entfallen.

In [3]:
rng = np.random.default_rng(SEED)
estimates = np.array([0.2, 0.1])
explore = rng.random() < 0.1
ad = int(rng.integers(2)) if explore else int(rng.choice(np.flatnonzero(estimates == estimates.max())))
print('Explore?', explore, '| Chosen ad:', ['A', 'B'][ad])

Explore? False | Chosen ad: A


## 3. Bandit-Verfahren versus sequenzielles Reinforcement Learning

Beide Ansätze können darauf abzielen, die gesamte erhaltene Belohnung zu maximieren.

Unser Bandit-Verfahren aktualisiert seine Schätzungen, ohne einen durch die gewählten Aktionen beeinflussten Zustand des Besuchers zu modellieren.

Sequenzielles Reinforcement Learning (RL) modelliert dagegen, wie eine Entscheidung spätere Situationen verändert.

Wenn ein Assistent nach einer fehlenden Präferenz zum Arbeitsort fragt, kostet dies zwar eine zusätzliche Interaktion, kann aber eine spätere Empfehlung verbessern.

Ein Assistent, der einem festgelegten Skript folgt, wurde nicht automatisch mit Reinforcement Learning trainiert.

## 4. Wie menschliches Feedback mit LLMs zusammenhängt

1. Texte liefern Zielwerte für die Vorhersage des nächsten Tokens. Beispielantworten liefern Zielwerte für das überwachte Feintuning (Supervised Fine-Tuning).

2. Menschen vergleichen Antworten. Ein separates Bewertungsmodell lernt diese Vergleiche mithilfe von überwachtem Lernen.

3. Anschliessend kann belohnungsbasiertes Training die Entscheidungsstrategie (Policy) des Sprachmodells aktualisieren.

RLHF bedeutet Reinforcement Learning from Human Feedback (Verstärkendes Lernen aus menschlichem Feedback).

Dieser klassische Trainingsablauf ist weder universell noch mit der Epsilon-Greedy-Strategie gleichzusetzen.

Durch gewöhnliches Prompting werden die Modellgewichte normalerweise nicht verändert.

## Evaluation und Interpretation

Mehr simulierte Klicks sind kein Beweis dafür, dass die Stellenanzeigen in der Realität besser zu den Stellensuchenden passen.

Ein Bandit-Verfahren kann eine ungeeignete Ersatzkennzahl (Proxy) optimieren. Auch menschliche Präferenzen garantieren nicht, dass eine Antwort wahr oder korrekt ist.

## Zusammenfassung und geschäftliche Auswirkungen

Durch das Ausprobieren verschiedener Optionen erhält man Rückmeldungen, die sonst fehlen würden.

Eine hohe durchschnittliche Erfolgsrate kann Misserfolge bei einzelnen Personen verbergen.

Deshalb sollte immer hinterfragt werden, ob die verwendete Belohnung tatsächlich einen nützlichen geschäftlichen Erfolg misst.

## Einschränkungen

Dies ist ein kleines Lehrbeispiel und kein Nachweis für die Leistungsfähigkeit im praktischen Einsatz.

Halte die Evaluation von der Modellanpassung und Modellauswahl getrennt.

Untersuche Fehler und die Entscheidungen der Benutzer, anstatt dich nur auf eine zusammenfassende Kennzahl zu verlassen.

## Weiterführende Übungen

- Was sagt uns ein ausbleibender Klick auf Anzeige A über die nicht angezeigte Anzeige B?

- Was verändert sich, nachdem ein Assistent eine Frage gestellt hat?

- Könnten mehr Klicks dazu führen, dass sich das geschäftliche Ergebnis verschlechtert?

## References

**Required reading:** Sutton & Barto (2018), *Reinforcement Learning: An Introduction*, second edition,
Chapter 2 opening and Sections 2.1-2.2. http://incompleteideas.net/book/the-book-2nd.html
**Instructor reference:** Ouyang et al. (2022), *Training language models to follow instructions with human feedback*.
https://arxiv.org/abs/2203.02155
Chapter 2 provides the feedback concepts, not a description of modern LLM training.

### Jupyter notebook - footer info

Include your name, submission date and a short description of your changes when submitting.